### Setting

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time
import imageio
import numpy as np
import pandas as pd
from PIL import Image
from thop import profile

import torch
import torch.nn as nn

from hex.model.tools import read_mode_config
from hex.model.framework.base_framework import baseframework

/mnt/dataset/vnwy44/miniconda3/envs/hex/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not torch.cuda.is_available():
    print("CUDA is not available. No GPU detected.")
else:
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs: {num_gpus}")

    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        props = torch.cuda.get_device_properties(i)
        total_mem_gb = props.total_memory / (1024 ** 3)

        print(f"GPU {i}: {gpu_name}")
        print(f"  Total memory: {total_mem_gb:.2f} GB")
        print(f"  Compute capability: {props.major}.{props.minor}")


Number of GPUs: 1
GPU 0: NVIDIA A100-SXM4-80GB
  Total memory: 79.15 GB
  Compute capability: 8.0


### Load Model

In [4]:
model_path = '../pretrained_models/EAI_real_world_2B/hex_ac100_300k_8gpu_state_query_history2/checkpoints/steps_300000_pytorch_model.pt'
infer_model = baseframework.from_pretrained(model_path)
infer_model = infer_model.to(torch.bfloat16)
infer_model = infer_model.to("cuda").eval()

instruction = ["Make a salad"]

05/17 [18:01:01] INFO     | >> [*] Loading from local checkpoint path                            ]8;id=92156;file:///mnt/dataset/vnwy44/code/HEX/hex/model/framework/share_tools.py\share_tools.py]8;;\:]8;id=842474;file:///mnt/dataset/vnwy44/code/HEX/hex/model/framework/share_tools.py#246\246]8;;\
                          `../pretrained_models/EAI_real_world_2B/hex_ac100_300k_8gpu_state_quer                   
                          y_history2/checkpoints/steps_300000_pytorch_model.pt`                                    

[EmbodimentRegistry] Saved registry to pretrained_models/hex/EAI_real_world_2B/hex_ac100_300k_8gpu_state_query_history2/embodiment_registry.json
Total number of DiT parameters:  122441736


In [5]:
# Compute the total number of parameters.
total_params = sum(p.numel() for p in infer_model.parameters())

# Compute the number of trainable parameters.
trainable_params = sum(p.numel() for p in infer_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Approx. model size: {total_params * 4 / (1024**2):.2f} MB (assuming FP32)")

Total parameters: 2,456,023,401
Trainable parameters: 2,456,023,401
Approx. model size: 9368.99 MB (assuming FP32)


In [6]:
norm_stats = infer_model.norm_stats['unitree_g1_v2']
action_norm_stats = norm_stats["action"]
state_norm_stats = norm_stats["state"]

### Inference Time

In [12]:
class Normalizer:
    valid_modes = ["q99", "mean_std", "min_max", "binary"]

    def __init__(self, mode: str, statistics: dict):
        self.mode = mode
        self.statistics = statistics
        for key, value in self.statistics.items():
            self.statistics[key] = torch.tensor(value)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert isinstance(
            x, torch.Tensor
        ), f"Unexpected input type: {type(x)}. Expected type: {torch.Tensor}"

        # Normalize the tensor
        if self.mode == "q99":
            # Range of q99 is [-1, 1]
            q01 = self.statistics["q01"].to(x.dtype)
            q99 = self.statistics["q99"].to(x.dtype)

            # In the case of q01 == q99, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = q01 != q99
            normalized = torch.zeros_like(x)

            # Normalize the values where q01 != q99
            # Formula: 2 * (x - q01) / (q99 - q01) - 1
            normalized[..., mask] = (x[..., mask] - q01[..., mask]) / (
                q99[..., mask] - q01[..., mask]
            )
            normalized[..., mask] = 2 * normalized[..., mask] - 1

            # Set the normalized values to the original values where q01 == q99
            normalized[..., ~mask] = x[..., ~mask].to(x.dtype)

            # Clip the normalized values to be between -1 and 1
            normalized = torch.clamp(normalized, -1, 1)

        elif self.mode == "mean_std":
            # Range of mean_std is not fixed, but can be positive or negative
            mean = self.statistics["mean"].to(x.dtype)
            std = self.statistics["std"].to(x.dtype)

            # In the case of std == 0, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = std != 0
            normalized = torch.zeros_like(x)

            # Normalize the values where std != 0
            # Formula: (x - mean) / std
            normalized[..., mask] = (x[..., mask] - mean[..., mask]) / std[..., mask]

            # Set the normalized values to the original values where std == 0
            normalized[..., ~mask] = x[..., ~mask].to(x.dtype)

        elif self.mode == "min_max":
            # Range of min_max is [-1, 1]
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)

            # In the case of min == max, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = min != max
            normalized = torch.zeros_like(x)

            # Normalize the values where min != max
            # Formula: 2 * (x - min) / (max - min) - 1
            normalized[..., mask] = (x[..., mask] - min[..., mask]) / (
                max[..., mask] - min[..., mask]
            )
            normalized[..., mask] = 2 * normalized[..., mask] - 1

            # Set the normalized values to the original values where min == max
            # normalized[..., ~mask] = x[..., ~mask].to(x.dtype)
            # Set the normalized values to 0 where min == max
            normalized[..., ~mask] = 0

        elif self.mode == "scale":
            # Range of scale is [0, 1]
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)
            abs_max = torch.max(torch.abs(min), torch.abs(max))
            mask = abs_max != 0
            normalized = torch.zeros_like(x)
            normalized[..., mask] = x[..., mask] / abs_max[..., mask]
            normalized[..., ~mask] = 0

        elif self.mode == "binary":
            # Range of binary is [0, 1]
            normalized = (x > 0.5).to(x.dtype)
        else:
            raise ValueError(f"Invalid normalization mode: {self.mode}")

        return normalized

    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        assert isinstance(
            x, torch.Tensor
        ), f"Unexpected input type: {type(x)}. Expected type: {torch.Tensor}"
        if self.mode == "q99":
            q01 = self.statistics["q01"].to(x.dtype)
            q99 = self.statistics["q99"].to(x.dtype)
            return (x + 1) / 2 * (q99 - q01) + q01
        elif self.mode == "mean_std":
            mean = self.statistics["mean"].to(x.dtype)
            std = self.statistics["std"].to(x.dtype)
            return x * std + mean
        elif self.mode == "min_max":
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)
            return (x + 1) / 2 * (max - min) + min
        elif self.mode == "binary":
            return (x > 0.5).to(x.dtype)
        else:
            raise ValueError(f"Invalid normalization mode: {self.mode}")

In [7]:
step = 0

video_path = "./dataset/episode_000533.mp4"
reader = imageio.get_reader(video_path)
frame = reader.get_data(step)   
reader.close()
pil_img = Image.fromarray(frame)    # rgb
batch_images = [[pil_img]]

normalizer_state = Normalizer('q99', state_norm_stats)
normalizer_action = Normalizer('q99', action_norm_stats)

df = pd.read_parquet("./dataset/episode_000533.parquet")
state = np.stack([np.asarray(x, dtype=np.float32) for x in df['observation.state'][step: step+1]])
state = torch.from_numpy(state)
state = normalizer_state.forward(state).unsqueeze(0)
state = np.array(state)

tags=["unitree_g1_v2"]

In [8]:
n_warmup = 5
n_runs = 10

with torch.no_grad():
   # warmup
    for _ in range(n_warmup):
        action_pred: np.ndarray = infer_model.predict_action(batch_images, instruction, state, tags=tags)['normalized_actions'][0][:, :32]
        unnorm_action_pred = normalizer_action.inverse(torch.tensor(action_pred))
        unnorm_action_pred = unnorm_action_pred.numpy()
        torch.cuda.synchronize()

    # real timing
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            torch.cuda.synchronize()
            t0 = time.perf_counter()

            action_pred: np.ndarray = infer_model.predict_action(batch_images, instruction, state, tags=tags)['normalized_actions'][0][:, :32]
            unnorm_action_pred = normalizer_action.inverse(torch.tensor(action_pred))
            unnorm_action_pred = unnorm_action_pred.numpy()

            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append(t1 - t0)

    avg = sum(times) / len(times)
    p50 = sorted(times)[len(times)//2]
    print(f"Inference time per forward on {gpu_name}: {avg*1000:.2f} ms (median {p50*1000:.2f} ms) over {n_runs} runs")


Inference time per forward on NVIDIA A100-SXM4-80GB: 143.08 ms (median 143.09 ms) over 10 runs


### FLOPS

In [9]:
class PredictActionWrapper(nn.Module):
    def __init__(self, infer_model):
        super().__init__()
        self.infer_model = infer_model

    def forward(self, batch_images, instruction, state, tags):
        out = self.infer_model.predict_action(batch_images, instruction, state, tags)
        actions = out["normalized_actions"] 
        return actions

In [10]:
wrapped = PredictActionWrapper(infer_model).to("cuda")
wrapped.eval()

PredictActionWrapper(
  (infer_model): HEX(
    (qwen_vl_interface): _QWen3_VL_Interface(
      (model): Qwen3VLForConditionalGeneration(
        (model): Qwen3VLModel(
          (visual): Qwen3VLVisionModel(
            (patch_embed): Qwen3VLVisionPatchEmbed(
              (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
            )
            (pos_embed): Embedding(2304, 1024)
            (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
            (blocks): ModuleList(
              (0-23): 24 x Qwen3VLVisionBlock(
                (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (attn): Qwen3VLVisionAttention(
                  (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                  (proj): Linear(in_features=1024, out_features=1024, bias=True)
                )
                (mlp): Qwen3VLVisionMLP(
                  (linear

In [11]:
macs, params = profile(
    wrapped, 
    inputs=(batch_images, instruction, state, tags),
)

print(f"MACs:  {macs/1e9:.2f} GMac")
print(f"Params:{params/1e6:.2f} M")
print(f"Approx FLOPs (≈2*MACs): {2*macs/1e9:.2f} GFLOPs")


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
MACs:  1111.99 GMac
Params:2419.64 M
Approx FLOPs (≈2*MACs): 2223.97 GFLOPs
